# Spotify Music Recommendation System
## Notebook 03 — Data Cleaning & Preprocessing

**Purpose:** Fix all known data quality issues found in notebook 01 and save clean datasets to `data/processed/`.

## Table of Contents
1. [Setup & Imports](#1-setup--imports)
2. [Load Raw Data](#2-load-raw-data)
3. [Fix 1 — Clean song_and_artists](#3-fix-1--clean-song_and_artists)
4. [Fix 2 — Parse artist list strings](#4-fix-2--parse-artist-list-strings)
5. [Fix 3 — Handle hidden-empty genres](#5-fix-3--handle-hidden-empty-genres)
6. [Fix 4 — release_date and type fixes](#6-fix-4--release_date-and-type-fixes)
7. [Fix 5 — Duration and outlier flags](#7-fix-5--duration-and-outlier-flags)
8. [Final Snapshot](#8-final-snapshot)
9. [Save Cleaned Datasets](#9-save-cleaned-datasets)
10. [Summary](#10-summary)

## 1. Setup & Imports

In [12]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)

# Make src/ importable from notebooks/
sys.path.insert(0, str(Path('..').resolve()))

from src.data.load import load_raw
from src.data.preprocess import (
    parse_artists,
    parse_genres,
    clean_song_artist,
    add_derived_columns,
    add_genre_columns,
    remove_short_tracks,
)

OUTPUT_DIR = Path('../data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('src modules loaded.')
print('Output directory ready:', OUTPUT_DIR.resolve())

src modules loaded.
Output directory ready: C:\Users\Victus\Music\Datascience\Projects\Recommendation_system\spotify_recommendation\data\processed


## 2. Load Raw Data

In [13]:
raw = load_raw()   # from src.data.load — single call, no hard-coded paths

data       = raw['data']
data_genre = raw['data_genre']
genre_df   = raw['genre']
year_df    = raw['year']
song_df    = raw['song_artist']

print('Loaded raw datasets:')
for name, df in [('data', data), ('data_genre', data_genre), ('genre_df', genre_df),
                  ('year_df', year_df), ('song_df', song_df)]:
    print(f'  {name:<15} {df.shape}')

Loaded raw datasets:
  data            (170653, 19)
  data_genre      (28680, 16)
  genre_df        (2973, 14)
  year_df         (100, 14)
  song_df         (2420, 5)


## 3. Fix 1 — Clean song_and_artists

**Problems:** 16 duplicate rows, 10 null `Singer/Artists`, 3 null `Album/Movie`, and `User-Rating` stored as a string format `'9.4/10'`.

`clean_song_artist()` from `src.data.preprocess` handles all four issues in one call.

In [14]:
print('Before cleaning:')
print('  Rows      :', len(song_df))
print('  Duplicates:', song_df.duplicated().sum())
print('  Nulls     :', song_df.isnull().sum().to_dict())

song_clean = clean_song_artist(song_df)   # src function

print('\nAfter cleaning:')
print('  Rows      :', len(song_clean))
print('  Duplicates:', song_clean.duplicated().sum())
print('  Nulls     :', song_clean.isnull().sum().to_dict())
print('  Rating range:', song_clean['rating'].min(), '\u2013', song_clean['rating'].max())
song_clean.head(3)

Before cleaning:
  Rows      : 2420
  Duplicates: 16
  Nulls     : {'Song-Name': 0, 'Singer/Artists': 10, 'Genre': 0, 'Album/Movie': 3, 'User-Rating': 0}

After cleaning:
  Rows      : 2404
  Duplicates: 0
  Nulls     : {'song_name': 0, 'artist': 0, 'genre': 0, 'album': 0, 'user_rating_raw': 0, 'rating': 0}
  Rating range: 5.5 – 9.9


,song_name,artist,genre,album,user_rating_raw,rating
0,Aankh Marey,"Kumar Sanu, Mika Singh, Neha Kakkar",BollywoodDance,Simmba,8.8/10,8.8
1,Coca Cola,"Neha Kakkar, Tony Kakkar",BollywoodDanceRomantic,Luka Chuppi,9.0/10,9.0
2,Apna Time Aayega,Ranveer Singh,BollywoodDance,Gully Boy,9.7/10,9.7


## 4. Fix 2 — Parse artist list strings

In `data.csv`, the `artists` column is stored as `"['Artist A', 'Artist B']"`.
`parse_artists()` from `src.data.preprocess` uses `ast.literal_eval()` to convert it to a real list.

> **Note:** `artists` in `data_w_genres.csv` is already a plain string — no parsing needed there.

In [15]:
# parse_artists() is imported from src — no need to redefine it here
data['artists_list']  = data['artists'].apply(parse_artists)
data['artists_clean'] = data['artists_list'].apply(lambda x: ', '.join(x))
data['n_artists']     = data['artists_list'].apply(len)

print('Parsing done.')
print(f'Single-artist tracks : {(data["n_artists"] == 1).sum():,}')
print(f'Multi-artist tracks  : {(data["n_artists"] > 1).sum():,}')
print()
data[['artists', 'artists_clean', 'n_artists']].head(5)

Parsing done.
Single-artist tracks : 136,175
Multi-artist tracks  : 34,478



,artists,artists_clean,n_artists
0,"['Sergei Rachmaninoff', 'James Levine', 'Berli...","Sergei Rachmaninoff, James Levine, Berliner Ph...",3
1,['Dennis Day'],Dennis Day,1
2,['KHP Kridhamardawa Karaton Ngayogyakarta Hadi...,KHP Kridhamardawa Karaton Ngayogyakarta Hadini...,1
3,['Frank Parker'],Frank Parker,1
4,['Phil Regan'],Phil Regan,1


## 5. Fix 3 — Handle hidden-empty genres

In `data_w_genres.csv`, the `genres` column is a Python list string.
Rows with `'[]'` are **not null** but have no genre info.
`add_genre_columns()` from `src.data.preprocess` parses all rows and adds helper columns.

In [16]:
data_genre = add_genre_columns(data_genre)   # src function

total = len(data_genre)
has   = data_genre['has_genre'].sum()
empty = total - has
print(f'Total artists          : {total:,}')
print(f'Artists WITH genres    : {has:,}  ({has/total*100:.1f}%)')
print(f'Artists WITHOUT genres : {empty:,}  ({empty/total*100:.1f}%)  <- these had [] string')
print()
data_genre[['artists', 'genres', 'genres_clean', 'n_genres', 'has_genre']].head(6)

Total artists          : 28,680
Artists WITH genres    : 18,823  (65.6%)
Artists WITHOUT genres : 9,857  (34.4%)  <- these had [] string



,artists,genres,genres_clean,n_genres,has_genre
0,"""Cats"" 1981 Original London Cast",['show tunes'],show tunes,1,True
1,"""Cats"" 1983 Broadway Cast",[],,0,False
2,"""Fiddler On The Roof” Motion Picture Chorus",[],,0,False
3,"""Fiddler On The Roof” Motion Picture Orchestra",[],,0,False
4,"""Joseph And The Amazing Technicolor Dreamcoat""...",[],,0,False
5,"""Joseph And The Amazing Technicolor Dreamcoat""...",[],,0,False


In [17]:
# Also fix the 1 empty-genre row in data_by_genres.csv
problem = (genre_df['genres'] == '[]').sum()
print(f'data_by_genres: {problem} row has genres=[] \u2014 dropping it')
genre_df_clean = genre_df[genre_df['genres'] != '[]'].reset_index(drop=True)
print(f'Rows after drop: {len(genre_df_clean)}')

data_by_genres: 1 row has genres=[] — dropping it
Rows after drop: 2972


## 6. Fix 4 — release_date and type fixes

- `release_date` has 3 formats — extract year cleanly, fall back to `year` column
- Cast `explicit` to bool, `key` and `mode` to int
- Drop the constant `mode` column from `data_by_year` (always equals 1)

`add_derived_columns()` from `src.data.preprocess` handles all of the above in one call.

In [18]:
data = add_derived_columns(data)   # src function — adds release_year, duration_min, flag_*

year_df_clean = year_df.drop(columns=['mode'])

print('Type fixes and derived columns added.')
print('year_df mode column dropped. Remaining columns:', year_df_clean.columns.tolist())
print()
match = (data['year'] == data['release_year']).mean()
print(f'year == release_year agreement: {match*100:.1f}%')

Type fixes and derived columns added.
year_df mode column dropped. Remaining columns: ['year', 'acousticness', 'danceability', 'duration_ms', 'energy', 'instrumentalness', 'liveness', 'loudness', 'speechiness', 'tempo', 'valence', 'popularity', 'key']

year == release_year agreement: 100.0%


## 7. Fix 5 — Duration and outlier flags

Outlier flags (`flag_short`, `flag_long`, `flag_zero_pop`) were added by `add_derived_columns()`.
We now remove tracks shorter than 1 minute — they are interludes or sound effects, not songs.

In [19]:
print('Outlier flags (added by add_derived_columns):')
print(f'  Tracks < 1 min  : {data["flag_short"].sum():,}')
print(f'  Tracks > 30 min : {data["flag_long"].sum():,}')
print(f'  Tracks pop = 0  : {data["flag_zero_pop"].sum():,}')

data_clean = remove_short_tracks(data)   # src function
print(f'\nModelling dataset after removing < 1 min tracks: {len(data_clean):,}')

Outlier flags (added by add_derived_columns):
  Tracks < 1 min  : 1,613
  Tracks > 30 min : 110
  Tracks pop = 0  : 27,892

Modelling dataset after removing < 1 min tracks: 169,040


## 8. Final Snapshot

In [20]:
print('=== data_clean ===')
print(f'Shape: {data_clean.shape}')
print('Columns:', data_clean.columns.tolist())
print()
print('=== song_clean ===')
print(f'Shape: {song_clean.shape}')
print('Columns:', song_clean.columns.tolist())
print()
print('=== data_genre (genres parsed) ===')
print(f'Shape: {data_genre.shape}')
print(f'Artists with real genres: {data_genre["has_genre"].sum():,}')

=== data_clean ===
Shape: (169040, 27)
Columns: ['valence', 'year', 'acousticness', 'artists', 'danceability', 'duration_ms', 'energy', 'explicit', 'id', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'name', 'popularity', 'release_date', 'speechiness', 'tempo', 'artists_list', 'artists_clean', 'n_artists', 'release_year', 'duration_min', 'flag_short', 'flag_long', 'flag_zero_pop']

=== song_clean ===
Shape: (2404, 6)
Columns: ['song_name', 'artist', 'genre', 'album', 'user_rating_raw', 'rating']

=== data_genre (genres parsed) ===
Shape: (28680, 20)
Artists with real genres: 18,823


## 9. Save Cleaned Datasets

In [21]:
data_clean.to_csv(OUTPUT_DIR / 'data_clean.csv', index=False)
song_clean.to_csv(OUTPUT_DIR / 'song_and_artists_clean.csv', index=False)
year_df_clean.to_csv(OUTPUT_DIR / 'data_by_year_clean.csv', index=False)
genre_df_clean.to_csv(OUTPUT_DIR / 'data_by_genres_clean.csv', index=False)

# Save artist-genre mapping (only artists that have genres)
artist_genre_map = (
    data_genre[data_genre['has_genre']][['artists', 'genres_clean', 'n_genres']]
    .reset_index(drop=True)
)
artist_genre_map.to_csv(OUTPUT_DIR / 'artist_genres.csv', index=False)

print('Saved to data/processed/:')
for f in sorted(OUTPUT_DIR.iterdir()):
    if not f.name.startswith('.'):
        print(' ', f.name)

Saved to data/processed/:
  artist_genres.csv
  data_by_genres_clean.csv
  data_by_year_clean.csv
  data_clean.csv
  feature_columns.json
  feature_matrix.csv
  song_and_artists_clean.csv
  track_index.csv


## 10. Summary

| Fix # | Problem | Solution | Rows Affected |
|---|---|---|---|
| 1 | `song_and_artists` — 16 duplicates | `clean_song_artist()` drop_duplicates | −16 |
| 1 | `song_and_artists` — 13 null values | Fill with 'Unknown' | 13 |
| 1 | `User-Rating` string format | Parse to float | 2,420 |
| 2 | `artists` in `data.csv` — list strings | `parse_artists()` ast.literal_eval | 170,653 |
| 3 | `genres` in `data_w_genres` — hidden `[]` | `add_genre_columns()` + has_genre flag | 9,857 |
| 3 | `genres` in `data_by_genres` — 1 empty row | Drop row | −1 |
| 4 | `release_date` mixed formats | `add_derived_columns()` → release_year | 1,610 |
| 4 | `mode` in `data_by_year` — constant 1 | Drop column | — |
| 5 | Tracks < 1 minute | `remove_short_tracks()` | −1,613 |

**Next:** `04_feature_engineering.ipynb` — build the 14-dimensional audio feature vector.